# citegraph quickstart

Two ways to run the pipeline on a folder of PDFs:

1. **All at once** — `pipe.run()` does every stage end-to-end.
2. **Step by step** — call each stage method, inspect the artifact on disk, decide whether to continue.

Both modes share the same `out_dir`, so you can mix and match: kick off `run()` once and then re-run an individual stage to iterate on it.

Set `GOOGLE_API_KEY` in your environment (or in a `.env` file in the current working directory) before running the cells below.

In [1]:
import citegraph, inspect
print(inspect.getfile(citegraph)) 

/Users/yabra/repos/citegraph/src/citegraph/__init__.py


In [2]:
from pathlib import Path

import pandas as pd

from citegraph import Pipeline

PDF_DIR = Path("/Users/yabra/Dropbox/Consultoria/Maria/data_papers4/")  # adapt to your project layout
OUT_DIR = Path("../out")

pipe = Pipeline(pdf_dir=PDF_DIR, 
                out_dir=OUT_DIR,
                model='gemini-3.1-flash-lite',
                enrich=False,
                recursive=True,
                show_progress=True)

## Mode A — run the whole pipeline

Use this when you trust the defaults and just want the three CSVs.
Skip this section entirely if you'd rather drive the pipeline stage-by-stage in Mode B.

In [ ]:
result = pipe.run()
print(
    f"papers={len(result.papers)}, "
    f"references={len(result.references)}, "
    f"edges={len(result.graph)}"
)

In [ ]:
result.papers.head()

In [ ]:
result.references.head()

## Mode B — progressive, one stage at a time

Each stage method writes its output to `OUT_DIR` and reads its input from disk if you call it with no arguments. So you can stop after any cell, eyeball the artifact, edit it by hand if you want, and only continue when you're happy.

The pipeline layout under `OUT_DIR`:

```
out/
  markdown/                 # stage 1: docling output, one .md per PDF
  metadata/<paper>.json     # stage 2: per-paper Gemini metadata
  references/<paper>.json   # stage 3: per-paper Gemini references
  papers.csv                # stage 2 aggregated
  references_raw.csv        # stage 3 aggregated
  references.csv            # stage 4 deduplicated
  citation_graph.csv        # stage 4 edges
```

Re-running a stage reuses any per-paper cache files it already wrote, so iterating is cheap.

### Stage 1 — PDFs ➜ markdown

`docling` is slow on first run; cached markdown is reused on subsequent calls. Pass `overwrite_markdown=True` on the `Pipeline` constructor if you need to force a re-conversion.

In [5]:
markdown_paths = pipe.convert_pdfs()
print(f"{len(markdown_paths)} markdown files in {pipe.layout.markdown_dir}")
for p in markdown_paths[:5]:
    print(" ", p.name)

*ERR* --- *ERR*
*ERR* Table is not square! *ERR*
*Padding to square...*
ERR#: COULD NOT CONVERT TO RS THIS TABLE TO COMPUTE SPANS
*ERR* --- *ERR*
*ERR* Table is not square! *ERR*
*Padding to square...*
*ERR* --- *ERR*
*ERR* Table is not square! *ERR*
*Padding to square...*
ERR#: COULD NOT CONVERT TO RS THIS TABLE TO COMPUTE SPANS
*ERR* --- *ERR*
*ERR* Table is not square! *ERR*
*Padding to square...*
*ERR* --- *ERR*
*ERR* Table is not square! *ERR*
*Padding to square...*
*ERR* --- *ERR*
*ERR* Table is not square! *ERR*
*Padding to square...*
*ERR* --- *ERR*
*ERR* Table is not square! *ERR*
*Padding to square...*


2026-05-10 20:49:18.059 ( 177.230s) [          3BA1FF]    doc_normalisation.h:448   WARN| found new `other` type: checkbox-unselected
2026-05-10 20:49:18.059 ( 177.230s) [          3BA1FF]    doc_normalisation.h:448   WARN| found new `other` type: checkbox-unselected
2026-05-10 20:49:18.062 ( 177.234s) [          3BA1FF]    doc_normalisation.h:448   WARN| found new `other` type: checkbox-unselected
2026-05-10 20:49:18.062 ( 177.234s) [          3BA1FF]    doc_normalisation.h:448   WARN| found new `other` type: checkbox-unselected
2026-05-10 20:49:18.062 ( 177.234s) [          3BA1FF]    doc_normalisation.h:448   WARN| found new `other` type: checkbox-unselected
2026-05-10 20:49:18.064 ( 177.235s) [          3BA1FF]    doc_normalisation.h:448   WARN| found new `other` type: checkbox-unselected
2026-05-10 20:49:18.064 ( 177.235s) [          3BA1FF]    doc_normalisation.h:448   WARN| found new `other` type: checkbox-unselected
2026-05-10 20:49:18.065 ( 177.237s) [          3BA1FF]    doc_

*ERR* --- *ERR*
*ERR* Table is not square! *ERR*
*Padding to square...*


*ERR* --- *ERR*
*ERR* Table is not square! *ERR*
*Padding to square...*
*ERR* --- *ERR*
*ERR* Table is not square! *ERR*
*Padding to square...*


*ERR* --- *ERR*
*ERR* Table is not square! *ERR*
*Padding to square...*
ERR#: COULD NOT CONVERT TO RS THIS TABLE TO COMPUTE SPANS
*ERR* --- *ERR*
*ERR* Table is not square! *ERR*
*Padding to square...*
*ERR* --- *ERR*
*ERR* Table is not square! *ERR*
*Padding to square...*
*ERR* --- *ERR*
*ERR* Table is not square! *ERR*
*Padding to square...*
*ERR* --- *ERR*
*ERR* Table is not square! *ERR*
*Padding to square...*
*ERR* --- *ERR*
*ERR* Table is not square! *ERR*
*Padding to square...*
ERR#: COULD NOT CONVERT TO RS THIS TABLE TO COMPUTE SPANS
*ERR* --- *ERR*
*ERR* Table is not square! *ERR*
*Padding to square...*
ERR#: COULD NOT CONVERT TO RS THIS TABLE TO COMPUTE SPANS
*ERR* --- *ERR*
*ERR* Table is not square! *ERR*
*Padding to square...*
*ERR* --- *ERR*
*ERR* Table is not square! *ERR*
*Padding to square...*
ERR#: COULD NOT CONVERT TO RS THIS TABLE TO COMPUTE SPANS
*ERR* --- *ERR*
*ERR* Table is not square! *ERR*
*Padding to square...*
ERR#: COULD NOT CONVERT TO RS THIS TABLE TO COMP

In [7]:
# # Spot-check the first markdown file before continuing.
# if markdown_paths:
#     print(markdown_paths[0].read_text()[:1500])

### Stage 2 — markdown ➜ paper metadata

Calls Gemini once per paper. Per-paper JSON is cached under `OUT_DIR/metadata/`, so re-running this cell after editing a single cached JSON file will pick up your edits and rewrite `papers.csv`.

In [3]:
papers = pipe.extract_paper_metadata()
papers.head()

Output()

,Title,Authors_List,Journal,Year,Authors,source_file,id
0,COMMUNICATION AND COOPERATION IN A COMMON-POOL...,"[Juan-Camilo Cardenas, T. K. Ahn, Elinor Ostrom]",Workshop in Political Theory and Policy Analysis,2003,"Juan-Camilo Cardenas, T. K. Ahn, Elinor Ostrom",Cardenas__Communication_and_Cooperation_in_a_C...,p-cardenas-2003-communication-and-cooperation-...
1,Discrimination in the Provision of Social Serv...,"[Juan-Camilo Cárdenas, Natalia Candelo, Alejan...",Research Network Working paper,2008,"Juan-Camilo Cárdenas, Natalia Candelo, Alejand...",Cardenas__Discrimination-in-the-Provision-of-S...,p-rdenas-2008-discrimination-in-the-provision-...
2,Dynamics of Rules and Resources: Three New Fie...,"[Juan-Camilo Cardenas, Marco Janssen, Francois...",Handbook on Experimental Economics and the Env...,2008,"Juan-Camilo Cardenas, Marco Janssen, Francois ...",Cardenas__Dynamics_of_Rules_and_Resources_Thre...,p-cardenas-2008-dynamics-of-rules-and-resource...
3,"Markets, Religion, Community Size, and the Evo...","[Joseph Henrich, Jean Ensminger, Richard McElr...",Science,2010,"Joseph Henrich, Jean Ensminger, Richard McElre...",Cardenas__MarketsReligionCommunity-2010.md,p-henrich-2010-markets-religion-community-size...
4,Rethinking Local Commons Dilemmas: Lessons fro...,[Juan-Camilo Cardenas],"Facultad de Estudios Ambientales y Rurales, Un...",2000,Juan-Camilo Cardenas,Cardenas__Rethinking_Local_Commons_Dilemmas_Le...,p-cardenas-2000-rethinking-local-commons-dilem...


### Stage 3 — markdown ➜ per-paper references

Also Gemini-driven, also cached per paper under `OUT_DIR/references/`. Output is the long-format `references_raw.csv` with one row per (citing_paper, raw_reference).

In [6]:
raw_refs = pipe.extract_paper_references()
print(f"{len(raw_refs)} raw references across {raw_refs['citing_id'].nunique()} papers")
raw_refs.head()

Model:                 gemini-3.1-flash-lite
Files to process:      3
Est. input tokens:     ~9,922
Est. output tokens:    ~15,450
Est. cost:             ~$0.0007 input + ~$0.0046 output = ~$0.0054 total
(Estimates are approximate; actual cost depends on response length.)
67 raw references across 3 papers


,Title,Authors_List,Journal,Year,Authors,citing_id
0,On the private provision of public goods,"[Bergstrom, T., Blume, L., Varian, H.]",J. Public Ecol.,1986,"Bergstrom, T., Blume, L., Varian, H.",p-cardenas-2002-economic-inequality-and-burden...
1,Unequal irrigators: Heterogeneity and commons ...,"[Bardhan, P., Dayton-Johnson, J.]",The drama of the commons,2002,"Bardhan, P., Dayton-Johnson, J.",p-cardenas-2002-economic-inequality-and-burden...
2,The voluntary provision of public goods under ...,"[Chan, K., Mestelman, S., Moir, R., Muller, R.A.]",Can. J. Ecol.,1996,"Chan, K., Mestelman, S., Moir, R., Muller, R.A.",p-cardenas-2002-economic-inequality-and-burden...
3,Heterogeneity and the voluntary provision of p...,"[Chan, K., Mestelman, S., Moir, R., Muller, R.A.]",Exp. Ecol.,1999,"Chan, K., Mestelman, S., Moir, R., Muller, R.A.",p-cardenas-2002-economic-inequality-and-burden...
4,Poverty and the environment: reversing the dow...,"[Durning, A.B.]",Worldwatch Paper No. 92,1989,"Durning, A.B.",p-cardenas-2002-economic-inequality-and-burden...


In [7]:
raw_refs

,Title,Authors_List,Journal,Year,Authors,citing_id
0,On the private provision of public goods,"[Bergstrom, T., Blume, L., Varian, H.]",J. Public Ecol.,1986,"Bergstrom, T., Blume, L., Varian, H.",p-cardenas-2002-economic-inequality-and-burden...
1,Unequal irrigators: Heterogeneity and commons ...,"[Bardhan, P., Dayton-Johnson, J.]",The drama of the commons,2002,"Bardhan, P., Dayton-Johnson, J.",p-cardenas-2002-economic-inequality-and-burden...
2,The voluntary provision of public goods under ...,"[Chan, K., Mestelman, S., Moir, R., Muller, R.A.]",Can. J. Ecol.,1996,"Chan, K., Mestelman, S., Moir, R., Muller, R.A.",p-cardenas-2002-economic-inequality-and-burden...
3,Heterogeneity and the voluntary provision of p...,"[Chan, K., Mestelman, S., Moir, R., Muller, R.A.]",Exp. Ecol.,1999,"Chan, K., Mestelman, S., Moir, R., Muller, R.A.",p-cardenas-2002-economic-inequality-and-burden...
4,Poverty and the environment: reversing the dow...,"[Durning, A.B.]",Worldwatch Paper No. 92,1989,"Durning, A.B.",p-cardenas-2002-economic-inequality-and-burden...
...,...,...,...,...,...,...
62,The speed of learning in noisy games,"[Bereby-Meyer Y, Roth A]",Am Econ Rev,2006,"Bereby-Meyer Y, Roth A",p-rdenas-2017-fragility-of-the-provision-of-lo...
63,Social networks and cooperation in hunter-gath...,"[Apicella CL, Marlowe FW, Fowler JH, Christaki...",Nature,2012,"Apicella CL, Marlowe FW, Fowler JH, Christakis NA",p-rdenas-2017-fragility-of-the-provision-of-lo...
64,Living with disturbance. Navigating Social-Eco...,"[Colding J, Elmquist T, Olsson P]",Cambridge Univ Press,2003,"Colding J, Elmquist T, Olsson P",p-rdenas-2017-fragility-of-the-provision-of-lo...
65,"Social capital, collective action, and adaptat...",[Adger WN],Econ Geogr,2003,Adger WN,p-rdenas-2017-fragility-of-the-provision-of-lo...


### Stage 4 — fuzzy dedup + citation graph

Pure-Python; no network. The dedup config (weights, threshold, year window) is what you'll most often want to tweak between iterations — pass a custom `DedupConfig` to `Pipeline(..., dedup_config=...)` and rerun this cell.

In [8]:
references, graph = pipe.deduplicate()
print(f"{len(raw_refs)} raw -> {len(references)} unique references, {len(graph)} edges")
references.head()

67 raw -> 67 unique references, 67 edges


,Authors,Journal,Title,Year
id,,,,
r-bergstrom-1986-on-the-private-provision-of-public-goods,"Bergstrom, T., Blume, L., Varian, H.",J. Public Ecol.,On the private provision of public goods,1986
r-bardhan-2002-unequal-irrigators-heterogeneity-and-in,"Bardhan, P., Dayton-Johnson, J.",The drama of the commons,Unequal irrigators: Heterogeneity and commons ...,2002
r-chan-1996-the-voluntary-provision-of-public-goods,"Chan, K., Mestelman, S., Moir, R., Muller, R.A.",Can. J. Ecol.,The voluntary provision of public goods under ...,1996
r-chan-1999-heterogeneity-and-the-voluntary-of-goods,"Chan, K., Mestelman, S., Moir, R., Muller, R.A.",Exp. Ecol.,Heterogeneity and the voluntary provision of p...,1999
r-durning-1989-poverty-and-the-environment-reversing,"Durning, A.B.",Worldwatch Paper No. 92,Poverty and the environment: reversing the dow...,1989


In [9]:
references

,Authors,Journal,Title,Year
id,,,,
r-bergstrom-1986-on-the-private-provision-of-public-goods,"Bergstrom, T., Blume, L., Varian, H.",J. Public Ecol.,On the private provision of public goods,1986
r-bardhan-2002-unequal-irrigators-heterogeneity-and-in,"Bardhan, P., Dayton-Johnson, J.",The drama of the commons,Unequal irrigators: Heterogeneity and commons ...,2002
r-chan-1996-the-voluntary-provision-of-public-goods,"Chan, K., Mestelman, S., Moir, R., Muller, R.A.",Can. J. Ecol.,The voluntary provision of public goods under ...,1996
r-chan-1999-heterogeneity-and-the-voluntary-of-goods,"Chan, K., Mestelman, S., Moir, R., Muller, R.A.",Exp. Ecol.,Heterogeneity and the voluntary provision of p...,1999
r-durning-1989-poverty-and-the-environment-reversing,"Durning, A.B.",Worldwatch Paper No. 92,Poverty and the environment: reversing the dow...,1989
...,...,...,...,...
r-berebymeyery-2006-the-speed-of-learning-in-noisy-games,"Bereby-Meyer Y, Roth A",Am Econ Rev,The speed of learning in noisy games,2006
r-apicellacl-2012-social-networks-and-cooperation-in,"Apicella CL, Marlowe FW, Fowler JH, Christakis NA",Nature,Social networks and cooperation in hunter-gath...,2012
r-coldingj-2003-living-with-disturbance-navigating,"Colding J, Elmquist T, Olsson P",Cambridge Univ Press,Living with disturbance. Navigating Social-Eco...,2003


In [10]:
graph


,citing_id,cited_id
0,p-cardenas-2002-economic-inequality-and-burden...,r-bergstrom-1986-on-the-private-provision-of-p...
1,p-cardenas-2002-economic-inequality-and-burden...,r-bardhan-2002-unequal-irrigators-heterogeneit...
2,p-cardenas-2002-economic-inequality-and-burden...,r-chan-1996-the-voluntary-provision-of-public-...
3,p-cardenas-2002-economic-inequality-and-burden...,r-chan-1999-heterogeneity-and-the-voluntary-of...
4,p-cardenas-2002-economic-inequality-and-burden...,r-durning-1989-poverty-and-the-environment-rev...
...,...,...
62,p-rdenas-2017-fragility-of-the-provision-of-lo...,r-berebymeyery-2006-the-speed-of-learning-in-n...
63,p-rdenas-2017-fragility-of-the-provision-of-lo...,r-apicellacl-2012-social-networks-and-cooperat...
64,p-rdenas-2017-fragility-of-the-provision-of-lo...,r-coldingj-2003-living-with-disturbance-naviga...
65,p-rdenas-2017-fragility-of-the-provision-of-lo...,r-wn-2003-social-capital-collective-action-and-to


### Stage 5 — optional CrossRef / OpenAlex enrichment

Skip unless you installed the `[crossref]` extra and want DOIs filled in. Construct a separate `Pipeline` with `enrich=True` (or set the flag on `pipe`) before calling.

In [12]:
pipe_enriched = Pipeline(pdf_dir=PDF_DIR, out_dir=OUT_DIR, enrich=True)
references = pipe_enriched.maybe_enrich()
references.head()

AttributeError: 'NoneType' object has no attribute 'get'

## Analysis — top-cited references

Works against either mode's output (the artifacts on disk are the same).
If you only ran Mode B, load the CSVs from disk first.

In [11]:
# Load from disk so this cell works regardless of which mode you ran above.
references = pd.read_csv(pipe.layout.references_csv, index_col="id")
graph = pd.read_csv(pipe.layout.graph_csv)

top = (
    graph.groupby("cited_id")
    .size()
    .sort_values(ascending=False)
    .head(10)
)
references.loc[top.index].assign(citations=top)

,Authors,Journal,Title,Year,citations
cited_id,,,,,
r-a-2011-safeguarding-food-security-in-volatile,Prakash A,Food and Agriculture Organization of the Unite...,Safeguarding Food Security in Volatile Global ...,2011,1
r-ostrom-2000-collective-action-and-the-evolution-of,"Ostrom, E.",J. Ecol. Perspect.,Collective action and the evolution of social ...,2000,1
r-joshi-2000-institutional-opportunities-and-in-the,"Joshi, N.N., Ostrom, E., Shivakoti, G.P., Lam,...",Asia-Pacific Journal of Rural Development,Institutional opportunities and constraints in...,2000,1
r-kochermg-2015-the-role-of-beliefs-trust-and-risk-in-to,"Kocher MG, Martinsson P, Matzat D, Wollbrant C",J Econ Psychol,"The role of beliefs, trust, and risk in contri...",2015,1
r-lam-1998-governing-irrigation-systems-in-nepal,"Lam, W.F.",ICS Press,Governing Irrigation Systems in Nepal: Institu...,1998,1
r-lansing-1991-priests-and-programmers-technologies-of,"Lansing, J.S.",Princeton University Press,Priests and Programmers: Technologies of Power...,1991,1
r-ledyard-1995-public-goods-a-survey-of-experimental,"Ledyard, J.O.",Handbook of Experimental Economics,Public goods: a survey of experimental research,1995,1
r-leonard-1985-divesting-natures-capital-the-political,"Leonard, H.J.",Holmes and Meier,Divesting Natures Capital-The Political Econom...,1985,1
r-leonard-1989-environment-and-the-poor-development-for,"Leonard, H.J.",Overseas Development Council,Environment and the Poor: Development Strategi...,1989,1
